# Demo 1: Vault en Kubernetes sobre Docker Desktop

Este notebook despliega un servidor de Vault en modo **dev** sobre el clúster de Kubernetes integrado en **Docker Desktop**, usando **Helm**.

> ⚠️ Uso exclusivamente educativo. El modo `dev` de Vault **no** es seguro para producción.

## Pasos

1. Verificar el clúster de Docker Desktop.
2. Crear el `namespace` `vault`.
3. Añadir el repo de Helm de HashiCorp.
4. Crear un archivo de valores adaptado a Docker Desktop.
5. Instalar/actualizar Vault con Helm.
6. Acceder a la UI y obtener el token raíz para usar en otros notebooks.


In [ ]:
%%bash
set -euo pipefail

echo "==> Comprobando cluster de Docker Desktop"
kubectl config current-context
kubectl get nodes -o wide

echo "==> Listando pods en todos los namespaces (vista rápida)"
kubectl get pods -A | head -n 20


In [ ]:
%%bash
set -euo pipefail

NAMESPACE=vault

echo "==> Creando namespace $NAMESPACE (si no existe)"
kubectl create namespace "$NAMESPACE" 2>/dev/null || echo "Namespace ya existe"

echo "==> Añadiendo/actualizando repo de Helm de HashiCorp"
helm repo add hashicorp https://helm.releases.hashicorp.com 2>/dev/null || true
helm repo update

cat > values-docker-desktop.yaml <<EOF
server:
  dataStorage:
    enabled: false
  dev:
    enabled: true
  service:
    type: ClusterIP
  readinessProbe:
    enabled: false
  livenessProbe:
    enabled: false
Injector:
  enabled: false
EOF

echo "==> Instalando / actualizando Vault con Helm"
helm upgrade --install vault hashicorp/vault \
  --namespace "$NAMESPACE" \
  -f values-docker-desktop.yaml

echo "==> Esperando pods de Vault"
kubectl get pods -n "$NAMESPACE"


## Acceder a la UI de Vault

En una **terminal separada**, ejecuta:

```bash
kubectl -n vault port-forward svc/vault 8200:8200
```

Luego abre `http://localhost:8200` en tu navegador. El token raíz de Vault en modo `dev` se puede obtener desde los logs:

```bash
kubectl -n vault logs statefulset/vault | head
```

Guarda ese token porque lo usarás en el siguiente notebook (`2_kubernetes_auth_method.ipynb`).
